In [ ]:
"""
House price prediction model for London properties.

This script loads historical property data, performs preprocessing and
feature engineering, trains a Random Forest regression model, evaluates
its performance, and predicts the estimated current price of a
hypothetical property.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor


In [ ]:
# Load the original dataset from CSV into a pandas DataFrame.
raw = pd.read_csv("kaggle_london_house_price_data_original.csv")
raw.columns


In [ ]:
# Selecting only the variables needed for this project.
columns_to_keep = [
    "outcode",
    "latitude",
    "longitude",
    "bathrooms",
    "bedrooms",
    "floorAreaSqM",
    "livingRooms",
    "propertyType",
    "history_date",
    "history_price",
    "saleEstimate_currentPrice"  # target
]

df = raw[columns_to_keep].copy()

df.head()


,outcode,latitude,longitude,bathrooms,bedrooms,floorAreaSqM,livingRooms,propertyType,history_date,history_price,saleEstimate_currentPrice
0,EC4A,51.517282,-0.110314,1.0,1.0,45.0,1.0,Purpose Built Flat,1995-01-02,830000,600000.0
1,EC4A,51.517282,-0.110314,NaN,NaN,NaN,NaN,Flat/Maisonette,1995-01-02,830000,600000.0
2,SW1P,51.495505,-0.132379,2.0,2.0,71.0,1.0,Flat/Maisonette,1995-01-03,249950,759000.0
3,SE5,51.478185,-0.092201,1.0,1.0,64.0,1.0,Flat/Maisonette,1995-01-03,32000,388000.0
4,N10,51.588774,-0.139599,1.0,4.0,137.0,2.0,End Terrace House,1995-01-03,133000,1261000.0


In [ ]:
df["history_date"] = pd.to_datetime(df["history_date"], errors="coerce")
df["history_year"] = df["history_date"].dt.year

current_year = 2025
df["year_since_sale"] = current_year - df["history_year"]

# Drop the raw datetime – RandomForest can't use it
df = df.drop("history_date", axis=1)


In [ ]:
df[["history_year", "year_since_sale"]].head()


,history_year,year_since_sale
0,1995,30
1,1995,30
2,1995,30
3,1995,30
4,1995,30


In [ ]:
"history_year" in df.columns, "year_since_sale" in df.columns


(True, True)

In [ ]:
df = pd.get_dummies(df, columns=["outcode", "propertyType"], drop_first=True)


In [ ]:
df = df.dropna()


In [ ]:
X = df.drop("saleEstimate_currentPrice", axis=1)
y = df["saleEstimate_currentPrice"]



In [ ]:
"history_year" in X.columns, "year_since_sale" in X.columns



(True, True)

In [ ]:
bool_cols = X.select_dtypes(include=["bool"]).columns
X[bool_cols] = X[bool_cols].astype(int)


In [ ]:
# Split the data into training and testing sets.
# 90% is used for training and 10% for evaluation.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.10,
    random_state=42,
    shuffle=True
)


In [ ]:
# Create the Random Forest regression model.
# n_estimators defines the number of decision trees in the forest.
rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.10,
    random_state=42
)

In [ ]:
# Train the model using the training data.
rf_model.fit(X_train, y_train)
# Use the trained model to predict prices for the test set.
y_pred = rf_model.predict(X_test)


In [ ]:
# Calculate common regression performance metrics.
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R² score:", r2)


MAE: 61397.37165961807
RMSE: 168961.8853143029
R² score: 0.9581510346193628


In [ ]:
results_sample = pd.DataFrame({
    "actual": y_test.values[:10],
    "predicted": y_pred[:10]
})
results_sample


,actual,predicted
0,702000.0,7.143496e+05
1,1132000.0,1.126380e+06
2,703000.0,7.153738e+05
3,810000.0,7.909725e+05
4,1426000.0,1.478490e+06
5,353000.0,3.645710e+05
6,1525000.0,1.795636e+06
7,384000.0,3.802098e+05
8,932000.0,9.805563e+05
9,247000.0,3.355633e+05


In [ ]:
feature_cols = X.columns          # all features the model expects
num_medians = X.median(numeric_only=True)  # typical values for numeric columns
current_year = 2025               # same year you used before


In [ ]:
def make_feature_row(
    outcode,
    property_type,
    bedrooms,
    bathrooms,
    floor_area_sqm=None,
    living_rooms=None,
    history_price=None,
    history_year=None
):
    """
    Build a single-row feature DataFrame for a hypothetical property.

    The returned row matches the structure and column order of the
    training feature matrix used by the machine-learning model. Default
    values are taken from the median of the training data, then replaced
    with any user-supplied property details.

    Args:
        outcode (str): Postal outcode, for example 'E15'.
        property_type (str): Property category such as 'Flat'.
        bedrooms (int): Number of bedrooms.
        bathrooms (int): Number of bathrooms.
        floor_area_sqm (float, optional): Floor area in square metres.
        living_rooms (int, optional): Number of living rooms.
        history_price (float, optional): Historical sale price if known.
        history_year (int, optional): Historical sale year if known.

    Returns:
        pandas.DataFrame: A one-row DataFrame formatted for prediction.
    """
    # Start with median values so all numeric features have valid defaults.
    row = num_medians.copy()

    # Replace default values with user inputs where provided.
    if "bedrooms" in row.index:
        row["bedrooms"] = bedrooms

    if "bathrooms" in row.index:
        row["bathrooms"] = bathrooms

    if "floorAreaSqM" in row.index and floor_area_sqm is not None:
        row["floorAreaSqM"] = floor_area_sqm

    if living_rooms is not None and "livingRooms" in row.index:
        row["livingRooms"] = living_rooms

    if history_price is not None and "history_price" in row.index:
        row["history_price"] = history_price

    # Update time-based features if a historical sale year is supplied.
    if history_year is not None:
        if "history_year" in row.index:
            row["history_year"] = history_year
        if "year_since_sale" in row.index:
            row["year_since_sale"] = current_year - history_year

    # Reset all outcode dummy columns to 0.
    for col in row.index:
        if col.startswith("outcode_"):
            row[col] = 0

    # Set the chosen outcode to 1 if it exists in the trained feature set.
    outcode_col = f"outcode_{outcode}"
    if outcode_col in row.index:
        row[outcode_col] = 1

    # Reset all property type dummy columns to 0.
    for col in row.index:
        if col.startswith("propertyType_"):
            row[col] = 0

    # Set the chosen property type to 1 if it exists in the trained feature set.
    prop_col = f"propertyType_{property_type}"
    if prop_col in row.index:
        row[prop_col] = 1

    # Convert the completed Series into a one-row DataFrame
    # with the correct column order expected by the model.
    row_df = pd.DataFrame([row], columns=feature_cols)

    return row_df

In [ ]:
def predict_price(
    outcode,
    property_type,
    bedrooms,
    bathrooms,
    floor_area_sqm=None,
    living_rooms=None,
    history_price=None,
    history_year=None,
    build_type="normal"
):
    """
    Predict the current estimated price of a property.

    This function prepares the property features using make_feature_row()
    and then passes the resulting row into the trained Random Forest
    model to generate a base price estimate. It then applies a simple
    premium factor for new-build or luxury properties.

    Args:
        outcode (str): Postal outcode of the property.
        property_type (str): Type of property.
        bedrooms (int): Number of bedrooms.
        bathrooms (int): Number of bathrooms.
        floor_area_sqm (float, optional): Floor area in square metres.
        living_rooms (int, optional): Number of living rooms.
        history_price (float, optional): Historical sale price if known.
        history_year (int, optional): Historical sale year if known.
        build_type (str, optional): One of "normal", "new_build", or "luxury".

    Returns:
        float: Predicted current property price in pounds sterling.
    """
    row_df = make_feature_row(
        outcode=outcode,
        property_type=property_type,
        bedrooms=bedrooms,
        bathrooms=bathrooms,
        floor_area_sqm=floor_area_sqm,
        living_rooms=living_rooms,
        history_price=history_price,
        history_year=history_year
    )

    # Predict the base price using the trained model.
    base_price = rf_model.predict(row_df)[0]

    # Apply a premium depending on the selected build type.
    premiums = {
        "normal": 0.00,
        "new_build": 0.12,
        "luxury": 0.20
    }

    build_type_clean = build_type.lower().strip()
    premium = premiums.get(build_type_clean, 0.00)

    final_price = base_price * (1 + premium)
    return final_price

In [ ]:
estimated = predict_price(
    outcode="E15",
    property_type="Flat",
    bedrooms=3,# ---- Console user interface for a single house price prediction ----

print("=== House Price Estimator ===")

# Ask the user for all property features.
user_outcode = input("Enter property outcode (e.g. E15): ").strip()
user_property_type = input(
    "Enter property type (e.g. Flat, Terraced, Semi-Detached, Detached): "
).strip()

# Required numeric inputs.
bedrooms_str = input("Enter number of bedrooms: ").strip()
bathrooms_str = input("Enter number of bathrooms: ").strip()

# Optional numeric inputs.
floor_area_str = input("Enter floor area in square metres (press Enter to skip): ").strip()
living_rooms_str = input("Enter number of living rooms (press Enter to skip): ").strip()
history_price_str = input("Enter historical sale price (press Enter to skip): ").strip()
history_year_str = input("Enter historical sale year (press Enter to skip): ").strip()

# Build type input.
build_choice = input("Enter building type (normal, new_build, luxury): ").lower().strip()

valid_build_types = ["normal", "new_build", "luxury"]
if build_choice not in valid_build_types:
    print("Invalid build type entered. Using 'normal' by default.")
    build_choice = "normal"

# Convert required numeric inputs.
try:
    user_bedrooms = int(bedrooms_str)
except ValueError:
    print("Invalid bedrooms value. Using 1 by default.")
    user_bedrooms = 1

try:
    user_bathrooms = int(bathrooms_str)
except ValueError:
    print("Invalid bathrooms value. Using 1 by default.")
    user_bathrooms = 1

# Convert optional numeric inputs.
if floor_area_str == "":
    user_floor_area = None
else:
    try:
        user_floor_area = float(floor_area_str)
    except ValueError:
        print("Invalid floor area entered. Ignoring this value.")
        user_floor_area = None

if living_rooms_str == "":
    user_living_rooms = None
else:
    try:
        user_living_rooms = int(living_rooms_str)
    except ValueError:
        print("Invalid living rooms value entered. Ignoring this value.")
        user_living_rooms = None

if history_price_str == "":
    user_history_price = None
else:
    try:
        user_history_price = float(history_price_str)
    except ValueError:
        print("Invalid historical sale price entered. Ignoring this value.")
        user_history_price = None

if history_year_str == "":
    user_history_year = None
else:
    try:
        user_history_year = int(history_year_str)
    except ValueError:
        print("Invalid historical sale year entered. Ignoring this value.")
        user_history_year = None

# Generate the prediction using user-entered values.
estimated = predict_price(
    outcode=user_outcode,
    property_type=user_property_type,
    bedrooms=user_bedrooms,
    bathrooms=user_bathrooms,
    floor_area_sqm=user_floor_area,
    living_rooms=user_living_rooms,
    history_price=user_history_price,
    history_year=user_history_year,
    build_type=build_choice
)

print("\n=== Estimated Price ===")
print(f"Estimated price ({build_choice}): £{estimated:,.0f}")
    bathrooms=1,
    floor_area_sqm=80,
    living_rooms=1,
    history_price=None,
    history_year=None,
    build_type=build_choice
)

print(f"Estimated price ({build_choice}): £{estimated:,.0f}")

Estimated price: £511,922
